# 합성곱 신경망 - CNN

1. 합성곱 파트 - convolution
    - 커널(kernenl) 또는 필터(filter)라고 불리는 정방향 행렬 ex(3x3, 5x5)을 촘촘하게 이동시키며 이미지와 곱연산을 수행
        - 스트라이드 : 필터를 몇 칸씩 이동시킬지 결정, 스트라이드가 커질 수록 출력되는 결과물의 크기는 작아짐
        - 패딩 : 중앙에 있는 픽셀은 수 없이 중복해서 필터와 곱해지며 강하게 반영되지만, 모서리, 가장자리에 있는 픽셀은 필터가 구석을 지날 갈 때 딱 한 번만 쓰이고 끝난다. 이처럼 외곽정보의 손실을 막기 위해 이미지 가장자리에 0를 채워넣는 기술

2. 활성화 함수 - ex) ReLU
    - 모델이 복잡한 패턴도 학습을 할 수 있도록 합성곱 연산 결과에 ReLU($f(x) = \max(0, x)$) 같은 활성화 함수를 적용

3. 풀링 레이어 - Pooling Layer
    - 합성곱(Convolution) 레이어를 거쳐 나온 특징 지도(Feature Map)의 크기를 줄이고 핵심 정보만 요약
        - 계산량 감소 및 과적합(Overfitting) 방지
            - 이미지 데이터의 가로·세로 크기를 대폭 줄여주기 때문에, 컴퓨터가 처리해야 할 연산량이 크게 감소. 이는 모델이 너무 세부적인 노이즈에 집착하다가 정작 중요한 패턴을 놓치는 '과적합' 현상을 방지
        - 공간적 불변성(Spatial Invariance) 확보
            - 사진 속 고양이가 중앙에 있든, 약간 왼쪽에 있든, 혹은 고개가 살짝 기울어져 있든 간에 컴퓨터가 똑같이 '고양이'로 알아볼 수 있게 만드는 능력을 줌.
    - 작동 방식
        - ex) 4x4 크기의 특징 지도
        - 1. 이미지를 2x2 크기 영역(총 4개)로 쪼갬
        - 2. 각 구역에 있는 4개의 숫자 중 가장 큰 숫자만 선택하고 나머지는 버림
        - 3. 결과적으로 가로, 세로 크기가 정확하게 줄어든 2x2 크기 결과물이 나옴
    - 해상도를 낮추어 중요한 특징만 선명하게 부각시키고, 컴퓨터의 연산 부담을 줄여주는 스마트한 데이터 압축 단계
4. 플래튼 - flatten
    - 3차원 형태의 이미지 특징 데이터를 1차원의 긴 줄(벡터)로 일렬로 늘어놓는 단계
        - 1. 합성곱, 풀링 단계에서는 공간적 특징을 유지해야 하므로 3차원 데이터를 다룸
        - 2. 완전 연결층 / 분류기 단계에서는 힌트를 종합해 정답을 맞추는 부분으로 1차원 데이터만 받음
        - 3. 플래튼은 이 두 파트 사이에서 "3차원 데이터를 1차원으로 모양을 바꾸어 연결해 주는 징검다리" 역할

5. 완전 연결 레이어 - Fully Connected Layer
    - 추출한 모든 힌트를 종합해 최종적인 결론(분류 또는 예측)을 내리는 판별기 역할을 하는 파트
        - 1. 가중치(Weight) 곱하기: 완전 연결 레이어는 각 단서의 중요도(가중치)를 곱함.
            - ex) '자동차'를 판별하는 뉴런은 '바퀴', '유리창' 단서에 높은 가중치를 곱하고, '날개' 단서에는 0에 가까운 낮은 가중치를 곱함
        - 2. 점수 합산: 각 뉴런이 계산한 점수들을 모두 더합니다.
        - 3. 최종 확률 변환 (Softmax): 나온 점수들을 확률(예: 자동차 92%, 비행기 5%, 배 3%)로 변환하여 가장 확률이 높은 '자동차'를 최종 정답으로 출력합니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


# 1. 데이터셋 로드 및 전처리 (MNIST 손글씨 데이터)
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

# 이미지 데이터 정규화 (0~255 사이의 값을 0~1 사이로 조정)
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

# CNN 입력을 위해 3차원 형태로 변환 (28x28 해상도, 1개의 채널(흑백))
train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# 2. CNN 모델 구축 (요청하신 5가지 단계 반영)
model = models.Sequential()

# [1단계 & 2단계] 합성곱(Convolution) + 패딩/스트라이드 + 활성화 함수(ReLU)
model.add(layers.Conv2D(
    filters=32,             # 32개의 다양한 커널(필터) 사용
    kernel_size=(3, 3),     # 3x3 크기의 정방향 행렬 필터
    strides=(1, 1),         # 스트라이드: 1칸씩 이동
    padding='same',         # 패딩: 외곽 정보 손실을 막기 위해 0을 채움 ('same'은 입력과 출력 크기를 같게 유지)
    activation='relu',      # 활성화 함수: 복잡한 패턴 학습을 위한 ReLU 적용
    input_shape=(28, 28, 1) # 입력 이미지 크기
))

# [3단계] 풀링 레이어 (Pooling Layer)
# 2x2 크기 영역에서 가장 큰 값만 추출(Max Pooling)하여 데이터 압축 및 과적합 방지
model.add(layers.MaxPooling2D(pool_size=(2, 2)))

# 추가 합성곱 + 풀링 층 (특징을 더 깊게 추출하기 위함)
model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))

# [4단계] 플래튼 (Flatten)
# 3차원의 특징 지도 데이터를 1차원의 긴 줄(벡터)로 일렬로 늘어놓는 징검다리 역할
model.add(layers.Flatten())

# [5단계] 완전 연결 레이어 (Fully Connected Layer)
# 추출한 힌트를 종합하여 가중치를 곱하고 점수를 합산하는 뉴런층
model.add(layers.Dense(64, activation='relu'))

# 최종 출력층: Softmax 함수를 사용하여 0~9까지의 숫자일 확률(총합 1)로 변환
model.add(layers.Dense(10, activation='softmax'))

# 3. 모델 컴파일 (학습 설정)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 4. 모델 학습 진행
print("--- 모델 학습 시작 ---")
model.fit(train_images, train_labels, epochs=3, batch_size=64)

# 5. 모델 평가
print("\n--- 모델 평가 시작 ---")
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"\n테스트 정확도: {test_acc * 100:.2f}%")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
--- 모델 학습 시작 ---
Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9524 - loss: 0.1584
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9856 - loss: 0.0461
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9901 - loss: 0.0321

--- 모델 평가 시작 ---
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9885 - loss: 0.0353

테스트 정확도: 98.85%
